# <font color="#418FDE" size="6.5" uppercase>**Spectral Hierarchical**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Use spectral clustering for non-convex cluster structures with suitable affinity construction. 
- Apply agglomerative clustering with linkage and stopping criteria matched to the data. 
- Interpret hierarchical structure and feature agglomeration outputs. 


## **1. Spectral Clustering Basics**

### **1.1. Spectral Clustering Overview**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_01_01.jpg?v=1783856148" width="250">



>* Uses similarities, not just raw positions.
>* Finds complex clusters through connection patterns.

>* Treats data as a similarity network.
>* Finds non-convex groups through connectivity patterns.

>* Similarity design determines spectral clustering success.
>* Graphs reveal complex clusters beyond center distance.



### **1.2. Affinity Matrix Design**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_01_02.jpg?v=1783856145" width="250">



>* Affinity design defines meaningful local similarity.
>* Good graphs capture real-world neighborhood relationships.

>* Balance local links to avoid blur, fragmentation.
>* Use scaled neighbor graphs to preserve shape.

>* Scale features and limit noisy connections.
>* Match affinity design to real similarity.



In [ ]:
#@title Python Code - Affinity Matrix Design

# This script illustrates local similarity choices.
# Affinity design shapes spectral clustering behavior.
# We compare dense and sparse graphs.

import numpy as np
import matplotlib.pyplot as plt

# Create two curved non convex groups.
rng = np.random.default_rng(7)
theta = np.linspace(0.0, np.pi, 60)
noise = rng.normal(0.0, 0.05, size=(60, 2))

# Build the first moon shape.
moon_a = np.column_stack((np.cos(theta), np.sin(theta)))
moon_a = moon_a + noise

# Build the second shifted moon.
moon_b = np.column_stack((1.0 - np.cos(theta), 0.5 - np.sin(theta)))
moon_b = moon_b + noise

# Combine points into one dataset.
X = np.vstack((moon_a, moon_b))
assert X.shape == (120, 2)

# Compute pairwise Euclidean distances.
diff = X[:, None, :] - X[None, :, :]
D = np.sqrt(np.sum(diff * diff, axis=2))

# Build a Gaussian affinity matrix.
sigma = 0.25
A_rbf = np.exp(-(D ** 2) / (2.0 * sigma ** 2))
np.fill_diagonal(A_rbf, 0.0)

# Build a nearest neighbor affinity.
k = 8
order = np.argsort(D, axis=1)
A_knn = np.zeros_like(D)

# Connect each point locally.
for i in range(X.shape[0]):
    nbrs = order[i, 1:k + 1]
    A_knn[i, nbrs] = 1.0

# Symmetrize the sparse graph.
A_knn = np.maximum(A_knn, A_knn.T)

# Summarize graph density briefly.
rbf_density = np.mean(A_rbf > 0.05)
knn_density = np.mean(A_knn > 0.0)
print(f"Points: {X.shape[0]}")
print(f"RBF density above 0.05: {rbf_density:.2f}")
print(f"kNN density: {knn_density:.2f}")
print("kNN better preserves local curved neighborhoods.")

# Plot data with local graph edges.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, A, title in [
    (axes[0], A_rbf, "Dense RBF affinity"),
    (axes[1], A_knn, "Sparse kNN affinity"),
]:
    ax.scatter(X[:, 0], X[:, 1], s=18, c="tab:blue")
    rows, cols = np.where(np.triu(A > (0.25 if title.startswith("Dense") else 0.0), 1))
    for r, c in zip(rows[:220], cols[:220]):
        ax.plot(X[[r, c], 0], X[[r, c], 1], color="gray", alpha=0.25, linewidth=0.6)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()



### **1.3. Affinity Matrix Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_01_03.jpg?v=1783856152" width="250">



>* Affinity defines meaningful relationships between data points.
>* Good choices reveal complex, domain-specific cluster shapes.

>* Affinity graphs define similarity in different ways.
>* Graph choice can reveal or distort clusters.

>* Balance affinity density, scale, and robustness.
>* Match similarity choices to real structure.



## **2. Agglomerative Linkage Choices**

### **2.1. Choosing Linkage Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_02_01.jpg?v=1783856154" width="250">



>* Linkage choice changes cluster shapes and memberships.
>* Single preserves chains; complete forms compact clusters.

>* Average linkage balances cohesion and local connections.
>* Ward favors compact clusters, struggles with irregular shapes.

>* Match linkage to data shape and purpose.
>* Distance metric and scale also matter.



### **2.2. Choosing Linkage Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_02_02.jpg?v=1783856158" width="250">



>* Linkage rules shape cluster merging behavior.
>* Single, complete, average suit different structures.

>* Match linkage to data similarity meaning.
>* Average or Ward often avoid extremes.

>* Choose linkage for domain meaning and scale.
>* Compare criteria for stable, interpretable clusters.



### **2.3. Stopping Criteria**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_02_03.jpg?v=1783856156" width="250">



>* Stopping rules decide where to cut.
>* Use cluster count or distance jumps.

>* Distance thresholds depend on scale and purpose.
>* Preprocess data and use domain judgment.

>* Choose cuts balancing detail and simplicity.
>* Match cluster usefulness to the task.



## **3. Interpreting Cluster Structure**

### **3.1. Reading Dendrograms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_03_01.jpg?v=1783856161" width="250">



>* Branch height shows cluster dissimilarity.
>* Dendrogram reveals grouping order and strength.

>* Cut height determines cluster number and size.
>* Large gaps suggest boundaries; interpret cautiously.

>* Branch shapes reveal subgroups and outliers.
>* Dendrograms support broad and detailed interpretation.



### **3.2. Reading Dendrograms**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_03_02.jpg?v=1783856166" width="250">



>* Merge height shows similarity and cluster strength.
>* Low early merges suggest clearer clusters.

>* Cut below large merge-height gaps.
>* Best cut depends on modeling choices.

>* Ignore leaf order; focus on merge heights.
>* Late-joining cases may need domain checking.



### **3.3. Feature Agglomeration Insights**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_22/Lecture_B/image_03_03.jpg?v=1783856163" width="250">



>* Groups similar variables to reveal shared themes.
>* Interpret clusters using patterns and domain knowledge.

>* Cut levels reveal redundancy or broader themes.
>* Check clusters against concepts, scale, and noise.

>* Grouped features simplify explanation and interpretation.
>* Clusters suggest patterns, not proven causes.



# <font color="#418FDE" size="6.5" uppercase>**Spectral Hierarchical**</font>


In this lecture, you learned to:
- Use spectral clustering for non-convex cluster structures with suitable affinity construction. 
- Apply agglomerative clustering with linkage and stopping criteria matched to the data. 
- Interpret hierarchical structure and feature agglomeration outputs. 

In the next Module (Module 23), we will go over 'None'